# Starter Notebook: Colab LoRA for Text-to-SVG

This notebook is a Colab-friendly adaptation of the course starter.

Goals:
- fine-tune a <=2B parameter instruction model with LoRA
- use only the provided `train.csv` and `test.csv`
- run on Google Colab with GPU
- generate a Kaggle-ready `submission.csv`

The workflow is:
1. switch Colab runtime to GPU
2. install packages and point the notebook at your CSV files
3. load and clean local data
4. format `(prompt, svg)` pairs for SFT
5. fine-tune a small Qwen model with LoRA
6. run batched inference on `test.csv`
7. validate SVG output and save submission

## Colab Setup and Constraints

### Required files
- `train.csv`: columns `id,prompt,svg`
- `test.csv`: columns `id,prompt`
- `sample_submission.csv`: columns `id,svg`

### Before you run
1. In Colab, switch runtime to `GPU`
2. Choose one data option:
   - upload `train.csv` and `test.csv` directly to `/content`
   - mount Google Drive and point `CONFIG["data_dir"]` to that folder
3. Run the install cell, then the setup cells in order

### Course constraints reflected here
- LoRA fine-tuning is required
- keep the base model at `<= 2B` parameters by default
- do not use external datasets
- keep inference efficient with short prompts, limited generation length, `model.eval()`, and `torch.no_grad()`

### Practical note
Start with the conservative defaults in `CONFIG` to make sure the full Colab pipeline runs end to end, then increase sample count or sequence length only if your GPU has headroom.

In [ ]:
# Run this in a fresh Colab runtime.
# !pip install -q datasets trl trxansformers accelerate peft bitsandbytes pandas lxml
!pip install -q datasets trl transformers accelerate peft bitsandbytes pandas lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 53.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset

try:
    import google.colab  # type: ignore
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Running in Colab: {IS_COLAB}")

Torch: 2.10.0+cu128
CUDA available: True
Running in Colab: True


In [ ]:
# Conservative defaults for typical Colab GPUs.
CONFIG = {
    "data_dir": "/content/drive/MyDrive/midterm/",
    "train_csv_name": "train.csv",
    "test_csv_name": "test.csv",
    "model_name": "unsloth/Qwen2.5-1.5B-Instruct",
    "max_seq_length": 8192,
    "prompt_max_chars": 320,
    "max_train_samples": 50000,
    "max_eval_samples": 200,
    "eval_size": 0.1,
    "lora_r": 16,
    "lora_alpha": 32,
    "learning_rate": 5e-5,
    "num_train_epochs": 2,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "inference_batch_size": 4,
    "max_new_tokens": 3072,
    "output_dir": "/content/qwen15b_svg_lora",
}

CONFIG

{'data_dir': '/content/drive/MyDrive/midterm/',
 'train_csv_name': 'train.csv',
 'test_csv_name': 'test.csv',
 'model_name': 'unsloth/Qwen2.5-1.5B-Instruct',
 'max_seq_length': 8192,
 'prompt_max_chars': 320,
 'max_train_samples': 50000,
 'max_eval_samples': 200,
 'eval_size': 0.1,
 'lora_r': 16,
 'lora_alpha': 32,
 'learning_rate': 5e-05,
 'num_train_epochs': 2,
 'per_device_train_batch_size': 1,
 'gradient_accumulation_steps': 16,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'eval_steps': 100,
 'save_steps': 200,
 'inference_batch_size': 4,
 'max_new_tokens': 3072,
 'output_dir': '/content/qwen15b_svg_lora'}

In [ ]:
# Colab data setup: choose upload or Google Drive.
USE_DRIVE = False
DRIVE_DATA_DIR = "/content/drive/MyDrive/midterm"
UPLOAD_IF_MISSING = False


if IS_COLAB and USE_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    CONFIG["data_dir"] = DRIVE_DATA_DIR


def maybe_upload_csvs(upload_if_missing=False):
    if not IS_COLAB or not upload_if_missing:
        return

    from google.colab import files  # type: ignore

    required = [CONFIG["train_csv_name"], CONFIG["test_csv_name"]]
    present = {name: (Path(CONFIG["data_dir"]) / name).exists() for name in required}
    if all(present.values()):
        return

    print("Upload train.csv and test.csv to the current Colab session.")
    files.upload()


maybe_upload_csvs(upload_if_missing=UPLOAD_IF_MISSING)


def resolve_data_path(filename):
    candidates = [
        Path(CONFIG["data_dir"]) / filename,
        Path(".") / filename,
        Path("/content") / filename,
    ]
    for path in candidates:
        if path.exists():
            return str(path)
    raise FileNotFoundError(
        f"Could not find {filename}. Checked: {[str(path) for path in candidates]}"
    )


TRAIN_CSV_PATH = resolve_data_path(CONFIG["train_csv_name"])
TEST_CSV_PATH = resolve_data_path(CONFIG["test_csv_name"])

print(f"Resolved train CSV: {TRAIN_CSV_PATH}")
print(f"Resolved test CSV: {TEST_CSV_PATH}")

Resolved train CSV: /content/drive/MyDrive/midterm/train.csv
Resolved test CSV: /content/drive/MyDrive/midterm/test.csv


In [ ]:
# Local CSV only: the course rules do not allow external datasets.
REQUIRED_TRAIN_COLUMNS = ["id", "prompt", "svg"]
REQUIRED_TEST_COLUMNS = ["id", "prompt"]

In [ ]:
DISALLOWED_TAGS_RE = re.compile(
    r"<(script|animate|animateTransform|animateMotion|set|foreignObject)[^>]*>.*?</\1>"
    r"|<(script|animate|animateTransform|animateMotion|set|foreignObject)[^/]*/?>",
    flags=re.IGNORECASE | re.DOTALL,
)

def clean_svg_for_training(svg_text):
    """去掉比赛不允许的标签，压缩空白"""
    if not svg_text or not isinstance(svg_text, str):
        return ""
    svg_text = DISALLOWED_TAGS_RE.sub("", svg_text)
    svg_text = re.sub(r"<!--.*?-->", "", svg_text, flags=re.DOTALL)  # 去注释
    svg_text = re.sub(r"\s+", " ", svg_text).strip()  # 压缩空白
    svg_text = re.sub(r">\s+<", "><", svg_text)  # 标签间空白
    if len(svg_text) > 16000:
        return ""
    return svg_text

def clean_prompt(text, max_chars):
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text[:max_chars]


def is_valid_svg(svg_text):
    if not svg_text or not isinstance(svg_text, str):
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False


def load_local_train_df(path, sample_size=None):
    df = pd.read_csv(path)
    missing = sorted(set(REQUIRED_TRAIN_COLUMNS) - set(df.columns))
    if missing:
        raise ValueError(f"Missing train columns: {missing}")

    df = df[REQUIRED_TRAIN_COLUMNS].copy()
    df["prompt"] = df["prompt"].map(lambda x: clean_prompt(x, CONFIG["prompt_max_chars"]))
    df["svg"] = df["svg"].fillna("").astype(str).str.strip()
    df["svg"] = df["svg"].map(clean_svg_for_training)  # ← 新增这一行
    df = df[df["prompt"].ne("")]
    df = df[df["svg"].map(is_valid_svg)]
    df = df[df["svg"].str.len() < 14000]  # ~8192 tokens 对应约 14000 chars

    if sample_size:
        df = df.sample(n=min(sample_size, len(df)), random_state=SEED)

    return df.reset_index(drop=True)


def load_local_test_df(path):
    df = pd.read_csv(path)
    missing = sorted(set(REQUIRED_TEST_COLUMNS) - set(df.columns))
    if missing:
        raise ValueError(f"Missing test columns: {missing}")

    df = df[REQUIRED_TEST_COLUMNS].copy()
    df["prompt"] = df["prompt"].map(lambda x: clean_prompt(x, CONFIG["prompt_max_chars"]))
    df = df[df["prompt"].ne("")]
    return df.reset_index(drop=True)

In [ ]:
train_df = load_local_train_df(
    TRAIN_CSV_PATH,
    sample_size=CONFIG["max_train_samples"],
)
test_df = load_local_test_df(TEST_CSV_PATH)

train_raw = Dataset.from_pandas(train_df, preserve_index=False)
splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

if CONFIG["max_eval_samples"]:
    eval_ds = eval_ds.select(range(min(CONFIG["max_eval_samples"], len(eval_ds))))

print(f"Clean train rows before split: {len(train_df)}")
print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
print(f"Test prompt rows: {len(test_df)}")
svg_char_lengths = train_df["svg"].str.len()
print(f"SVG chars — mean: {svg_char_lengths.mean():.0f}, "
      f"p90: {svg_char_lengths.quantile(0.9):.0f}, "
      f"p95: {svg_char_lengths.quantile(0.95):.0f}, "
      f"max: {svg_char_lengths.max():.0f}")
train_ds[0]

Clean train rows before split: 49987
Train rows: 44988
Eval rows: 200
Test prompt rows: 1000
SVG chars — mean: 2513, p90: 5093, p95: 6065, max: 13992


{'id': '34e1109239af19a1540e520a6a55e228',
 'prompt': "A simple silhouette of a person's head and torso in dark gray against a white background.",
 'svg': '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 200.0 200.0" height="200.0px" width="200.0px"><path fill="#333333" fill-opacity="1.0" filling="0" d="M100.0 20.0 A47.5 47.5 0.0 1 0 100.0 115.0 A47.5 47.5 0.0 0 0 100.0 20.0 Z M100.0 35.0 A32.5 32.5 0.0 1 1 100.0 100.0 A32.5 32.5 0.0 0 1 100.0 35.0 Z"></path><path fill="#333333" fill-opacity="1.0" filling="0" d="M60.78007888793945 55.0 L155.7800750732422 55.0 C163.28009033203125 55.0 163.28009033203125 55.0 163.28009033203125 62.5 L163.28009033203125 62.5 C163.28009033203125 70.0 163.28009033203125 70.0 155.7800750732422 70.0 L60.78007888793945 70.0 C53.28007888793945 70.0 53.28007888793945 70.0 53.28007888793945 62.5 L53.28007888793945 62.5 C53.28007888793945 55.0 53.28007888793945 55.0 60.78007888793945 55.0 Z"></path><path fill="#333333" fill-opacity="1.0" filling="0" d="M1

In [ ]:

SYSTEM_PROMPT = (
    "You generate SVG code from text descriptions. Rules:\n"
    "- Output ONLY a single <svg> element, nothing else\n"
    "- Canvas: width=\"256\" height=\"256\" viewBox=\"0 0 256 256\"\n"
    "- Always include xmlns=\"http://www.w3.org/2000/svg\"\n"
    "- Allowed elements: svg, g, path, rect, circle, ellipse, line, polyline, "
    "polygon, defs, use, symbol, clipPath, mask, linearGradient, radialGradient, "
    "stop, text, tspan, title, desc, style, pattern, marker, filter\n"
    "- Do NOT use: script, animate, foreignObject, event handlers\n"
    "- Keep SVG compact, under 16000 characters, max 256 paths"
)


def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}


train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print(train_text[0]["text"][:400])

Map:   0%|          | 0/44988 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

<|im_start|>system
You generate SVG code from text descriptions. Rules:
- Output ONLY a single <svg> element, nothing else
- Canvas: width="256" height="256" viewBox="0 0 256 256"
- Always include xmlns="http://www.w3.org/2000/svg"
- Allowed elements: svg, g, path, rect, circle, ellipse, line, polyline, polygon, defs, use, symbol, clipPath, mask, linearGradient, radialGradient, stop, text, tspan, 


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

print(f"Loaded model: {CONFIG['model_name']}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.17 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Loaded model: unsloth/Qwen2.5-1.5B-Instruct


In [ ]:
from transformers import TrainingArguments, TrainerCallback
from trl import SFTTrainer

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=use_fp16,
    bf16=use_bf16,
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
)

class DriveBackupCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        import shutil
        src = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
        dst = "/content/drive/MyDrive/midterm/latest_checkpoint"
        if os.path.exists(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"[Backup] checkpoint-{state.global_step} → Drive")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=True,
    args=training_args,
    callbacks=[DriveBackupCallback()],
)

import glob
checkpoints = sorted(glob.glob(os.path.join(CONFIG["output_dir"], "checkpoint-*")))
if not checkpoints:
    drive_ck = "/content/drive/MyDrive/midterm/latest_checkpoint"
    if os.path.exists(drive_ck):
        import shutil
        restore_to = os.path.join(CONFIG["output_dir"], "checkpoint-latest")
        shutil.copytree(drive_ck, restore_to, dirs_exist_ok=True)
        checkpoints = [restore_to]

resume_from = checkpoints[-1] if checkpoints else None
if resume_from:
    print(f"Resuming from {resume_from}")
train_result = trainer.train(resume_from_checkpoint=resume_from)
train_result

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/44988 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 44,988 | Num Epochs = 2 | Total steps = 5,624
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss,Validation Loss
100,0.681560,0.606768
200,0.494889,0.481141
300,0.439072,0.437051
400,0.396629,0.412799
500,0.399739,0.396550
600,0.402935,0.385856
700,0.373868,0.378470
800,0.367015,0.371535
900,0.359481,0.365421
1000,0.361060,0.361876


...

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


[Backup] checkpoint-5400 → Drive


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


[Backup] checkpoint-5600 → Drive
[Backup] checkpoint-5624 → Drive


TrainOutput(global_step=5624, training_loss=0.3417295063839189, metrics={'train_runtime': 30706.9281, 'train_samples_per_second': 2.93, 'train_steps_per_second': 0.183, 'total_flos': 6.763878852953825e+17, 'train_loss': 0.3417295063839189, 'epoch': 2.0})

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen2.5-1.5B-Instruct")
base_model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

adapter_path = "/content/drive/MyDrive/midterm/latest_checkpoint"
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()
print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

Model loaded successfully


In [ ]:
import re

SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)

def extract_svg(text):
    if "<|im_start|>assistant" in text:
        text = text.split("<|im_start|>assistant")[-1]
    match = SVG_REGEX.search(text)
    return match.group(0).strip() if match else ""

def post_process_svg(svg_text):
    if not svg_text:
        return ""
    svg_text = DISALLOWED_TAGS_RE.sub("", svg_text)
    if 'xmlns=' not in svg_text:
        svg_text = svg_text.replace('<svg', '<svg xmlns="http://www.w3.org/2000/svg"', 1)
    if 'viewBox=' not in svg_text:
        svg_text = svg_text.replace('<svg', '<svg viewBox="0 0 256 256"', 1)
    if len(svg_text) > 16000:
        return ""
    return svg_text

def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )

def build_chat_prompt(prompt):
    return (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

def generate_svg_batch(prompts, max_new_tokens=None):
    max_new_tokens = max_new_tokens or CONFIG["max_new_tokens"]
    chat_texts = [build_chat_prompt(prompt) for prompt in prompts]
    inputs = tokenizer(
        chat_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.05,
        )

    decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=False)
    results = []
    for text in decoded:
        svg = extract_svg(text)
        svg = post_process_svg(svg)
        results.append(svg)
    return results

print("All functions ready.")

All functions ready.


In [ ]:
SUBMISSION_PATH = "./submission.csv"
PROGRESS_PATH = "/content/drive/MyDrive/midterm/submission_progress.csv"

# 检查是否有之前的进度
import os
if os.path.exists(PROGRESS_PATH):
    existing = pd.read_csv(PROGRESS_PATH)
    rows = existing.to_dict('records')
    start_from = len(rows)
    invalid_count = sum(1 for r in rows if r['svg'] == fallback_svg(""))
    print(f"Resuming from {start_from}/{len(test_df)}, invalid so far: {invalid_count}")
else:
    rows = []
    start_from = 0
    invalid_count = 0
    print("Starting fresh")

t0 = time.time()
total = len(test_df)

for start in range(start_from, total, CONFIG["inference_batch_size"]):
    batch_df = test_df.iloc[start : start + CONFIG["inference_batch_size"]]
    batch_prompts = batch_df["prompt"].tolist()
    batch_svgs = generate_svg_batch(batch_prompts)

    for sample_id, prompt, svg in zip(batch_df["id"], batch_prompts, batch_svgs):
        if not is_valid_svg(svg):
            invalid_count += 1
            svg = fallback_svg(prompt)
        rows.append({"id": sample_id, "svg": svg})

    # 每个 batch 都保存到 Drive
    pd.DataFrame(rows).to_csv(PROGRESS_PATH, index=False)

    done = len(rows)
    elapsed = (time.time() - t0) / 60
    speed = (done - start_from) / elapsed if elapsed > 0 else 0
    eta = (total - done) / speed if speed > 0 else 0
    print(f"[{done}/{total}] invalid: {invalid_count} | {elapsed:.1f}min | {speed:.1f} samples/min | ETA: {eta:.1f}min")

# 最终保存
sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)
sub_df.to_csv(PROGRESS_PATH, index=False)

print(f"\nDone! Invalid: {invalid_count}/{total} ({invalid_count/total*100:.1f}%)")

Resuming from 920/1000, invalid so far: 766


KeyboardInterrupt: 

In [ ]:
eos_token_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
print("eos_token_id:", eos_token_id)

def generate_svg_batch(prompts, max_new_tokens=None):
    max_new_tokens = max_new_tokens or CONFIG["max_new_tokens"]
    chat_texts = [build_chat_prompt(prompt) for prompt in prompts]
    inputs = tokenizer(
        chat_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.05,
            eos_token_id=eos_token_id,
        )

    decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=False)
    results = []
    for text in decoded:
        svg = extract_svg(text)
        svg = post_process_svg(svg)
        results.append(svg)
    return results

print("Updated with eos_token_id")

eos_token_id: 151645
Updated with eos_token_id


In [ ]:
SUBMISSION_PATH = "./submission.csv"
PROGRESS_PATH = "/content/drive/MyDrive/midterm/submission_progress.csv"

import os
if os.path.exists(PROGRESS_PATH):
    existing = pd.read_csv(PROGRESS_PATH)
    rows = existing.to_dict('records')
    start_from = len(rows)
    invalid_count = sum(1 for r in rows if r['svg'] == fallback_svg(""))
    print(f"Resuming from {start_from}/{len(test_df)}")
else:
    rows = []
    start_from = 0
    invalid_count = 0

t0 = time.time()
total = len(test_df)

for start in range(start_from, total, CONFIG["inference_batch_size"]):
    batch_df = test_df.iloc[start : start + CONFIG["inference_batch_size"]]
    batch_prompts = batch_df["prompt"].tolist()
    batch_svgs = generate_svg_batch(batch_prompts)

    for sample_id, prompt, svg in zip(batch_df["id"], batch_prompts, batch_svgs):
        if not is_valid_svg(svg):
            invalid_count += 1
            svg = fallback_svg(prompt)
        rows.append({"id": sample_id, "svg": svg})

    pd.DataFrame(rows).to_csv(PROGRESS_PATH, index=False)

    done = len(rows)
    elapsed = (time.time() - t0) / 60
    speed = (done - start_from) / elapsed if elapsed > 0 else 0
    eta = (total - done) / speed if speed > 0 else 0
    print(f"[{done}/{total}] invalid: {invalid_count} | {elapsed:.1f}min | {speed:.1f} samples/min | ETA: {eta:.1f}min")

sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)
sub_df.to_csv(PROGRESS_PATH, index=False)
print(f"\nDone! Invalid: {invalid_count}/{total}")

Resuming from 1000/1000

Done! Invalid: 838/1000


In [ ]:
from google.colab import files
files.download("./submission.csv")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
shutil.copy("./submission.csv", "/content/drive/MyDrive/midterm/submission.csv")
print("Saved to Drive")

Saved to Drive


In [ ]:
# Optional Colab export helpers.
ARTIFACT_ZIP_PATH = "/content/qwen15b_svg_lora_artifacts.zip"

if IS_COLAB:
    import shutil
    from google.colab import files  # type: ignore

    shutil.make_archive(
        ARTIFACT_ZIP_PATH.removesuffix('.zip'),
        'zip',
        CONFIG["output_dir"],
    )
    print(f"Created artifact archive: {ARTIFACT_ZIP_PATH}")
    print("Uncomment the lines below to download outputs to your machine.")
    # files.download(SUBMISSION_PATH)
    # files.download(ARTIFACT_ZIP_PATH)
else:
    print(f"Submission saved at: {SUBMISSION_PATH}")
    print(f"Model artifacts saved at: {CONFIG['output_dir']}")

In [ ]:
sub_df = pd.read_csv("/content/drive/MyDrive/midterm/submission_progress.csv")
over_limit = sub_df["svg"].str.len() > 8000
print(f"超过 8000 字符: {over_limit.sum()} 条")
print(f"总条数: {len(sub_df)}")

超过 8000 字符: 0 条
总条数: 1000


In [ ]:
# 先打包成一个 zip 文件
import shutil
shutil.make_archive(
    "/content/drive/MyDrive/midterm/latest_checkpoint_zip",
    'zip',
    "/content/drive/MyDrive/midterm/latest_checkpoint"
)
print("Done! File at: /content/drive/MyDrive/midterm/latest_checkpoint_zip.zip")

Done! File at: /content/drive/MyDrive/midterm/latest_checkpoint_zip.zip


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

save_path = "/content/drive/MyDrive/midterm/qwen25-1.5b-instruct-base"

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
tokenizer.save_pretrained(save_path)

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model.save_pretrained(save_path)

print(f"Saved to {save_path}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/midterm/qwen25-1.5b-instruct-base


## Notes

- This version uses only the provided CSV files and does not rely on external training datasets.
- For Colab, the safest first run is the default config: small sample count, `max_seq_length = 1024`, and batched inference with `max_new_tokens = 256`.
- If your GPU is stable, try increasing `max_train_samples`, `max_seq_length`, or `inference_batch_size` one at a time.
- Keep a fixed seed, save logs, and record `invalid_count` for your report.
- After Colab finishes, download `submission.csv` and the zipped adapter directory so you can submit and archive your run.